In [78]:
!pip install torch plotly nbformat

In [79]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
inputs.shape

torch.Size([6, 3])

In [80]:
query = inputs[1] 
# Create a tensor with the same dimension of the inputs[1] with empty values
attn_scores_2 = torch.empty(inputs.shape[0])    

print("Query Vector (x^2):", query)
print("Attention Scores:\n", attn_scores_2)

# Create a product between the query and the other values on inputs to create a score for each value on inputs
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i)
    print(f"Dot product of query with x^{i+1}:", attn_scores_2[i])
    
print("Attention Scores:\n", attn_scores_2)

Query Vector (x^2): tensor([0.5500, 0.8700, 0.6600])
Attention Scores:
 tensor([0., 0., 0., 0., 0., 0.])
Dot product of query with x^1: tensor(0.9544)
Dot product of query with x^2: tensor(1.4950)
Dot product of query with x^3: tensor(1.4754)
Dot product of query with x^4: tensor(0.8434)
Dot product of query with x^5: tensor(0.7070)
Dot product of query with x^6: tensor(1.0865)
Attention Scores:
 tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [81]:
# Normalize the attention scores using softmax
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = torch.softmax(attn_scores_2, dim=0)
print("Attention Weights (Naive Softmax):\n", attn_weights_2_naive)
print("Sum of Attention Weights (Naive Softmax):", attn_weights_2_naive.sum())

Attention Weights (Naive Softmax):
 tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of Attention Weights (Naive Softmax): tensor(1.)


In [82]:
query = inputs[1]
context_vector_2 = torch.zeros(query.shape)

for i , x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2_naive[i] * x_i
    print(f"Context Vector after processing x^{i+1}:", context_vector_2)

Context Vector after processing x^1: tensor([0.0596, 0.0208, 0.1233])
Context Vector after processing x^2: tensor([0.1904, 0.2277, 0.2803])
Context Vector after processing x^3: tensor([0.3234, 0.4260, 0.4296])
Context Vector after processing x^4: tensor([0.3507, 0.4979, 0.4705])
Context Vector after processing x^5: tensor([0.4340, 0.5250, 0.4813])
Context Vector after processing x^6: tensor([0.4419, 0.6515, 0.5683])


In [83]:
attention_weights_all_input = torch.zeros(inputs.shape[0], inputs.shape[0])
for i, element in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attention_weights_all_input[i, j] = torch.dot(element, x_j)

# attention_weights_all_input = torch.softmax(attention_weights_all_input, dim=1)

print("Attention Weights for All Input Elements:\n", attention_weights_all_input)

Attention Weights for All Input Elements:
 tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [86]:
attention_weights_scores = inputs @ inputs.T
attention_weights_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [87]:
attention_weights_scores = torch.softmax(attention_weights_scores, dim=1)
attention_weights_scores

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [89]:
all_context_vectors = attention_weights_scores @ inputs
print("Context Vectors for All Input Elements:\n", all_context_vectors)

Context Vectors for All Input Elements:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
